# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [2]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [3]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [4]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [5]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [6]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [7]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

In [8]:
#only reserve needed columns to avoid a fat DF
p_states = patents.select(
    col("PATENT").alias("pid"),
    col("COUNTRY").alias("country"),
    col("POSTATE").alias("state")
).cache()


In [9]:
#This line selects the citation relationships and renames the columns to clearly represent the citing and cited patents.
c = citations.select(col("CITING").alias("citing"), col("CITED").alias("cited"))



In [10]:
# We join the citation table with the patent table to add state information for the cited patents.
# A left join is used because some cited patents do not exist in the patent table.
j1 = c.join(
    p_states.select(col("pid").alias("cited_pid"), col("country").alias("cited_country"), col("state").alias("cited_state")),
    on=(col("cited") == col("cited_pid")),
    how="left"
)



In [11]:
# Perform a second join to add state information for the citing patents.
# This allows us to obtain the state of both the citing and cited patents for each citation.
j2 = j1.join(
    p_states.select(col("pid").alias("citing_pid"), col("country").alias("citing_country"), col("state").alias("citing_state")),
    on=(col("citing") == col("citing_pid")),
    how="left"
)



In [12]:
# intermediate table
intermediate = j2.select("cited", "cited_state", "citing", "citing_state")
# intermediate.show(10)



In [13]:
# filter ruls: citing and cited both need to be (US and state not null)
filtered = intermediate.filter(
    (col("citing_state").isNotNull()) &
    (col("cited_state").isNotNull()) &
    (col("citing_country") == "US") &
    (col("cited_country") == "US")
)



In [14]:
#for each citing patent, count how many "cited_state == citing_state" it cited
self_counts = filtered.filter(col("cited_state") == col("citing_state")) \
    .groupBy("citing") \
    .agg(count("*").alias("self_state_cites"))



In [15]:
top10 = self_counts.orderBy(col("self_state_cites").desc(), col("citing").asc()).limit(10)

top10.show(10, truncate=False)

+-------+----------------+
|citing |self_state_cites|
+-------+----------------+
|5959466|125             |
|5983822|103             |
|6008204|100             |
|5952345|98              |
|5958954|96              |
|5998655|96              |
|5936426|94              |
|5739256|90              |
|5913855|90              |
|5925042|90              |
+-------+----------------+



In [16]:
# Join the top 10 results back with the patent table to retrieve the full patent information for each citing patent.
result = top10.join(
    patents,
    top10["citing"] == patents["PATENT"],
    how="inner"
)

# Rename the column to SAME_STATE to match the required output format.
result = result.withColumnRenamed("self_state_cites", "SAME_STATE")

# Reorder the columns so that all original patent fields appear first, followed by the SAME_STATE column.
cols = patents.columns + ["SAME_STATE"]

final_top10 = result.select(*cols).orderBy(col("SAME_STATE").desc(), col("PATENT").asc())

final_top10.show(10, truncate=False)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|PATENT |GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466|1999 |14515|1997   |US     |CA     |5310    |2      |NULL  |326   |4  |46    |159  |0       |1.0     |NULL   |0.6186  |NULL    |4.8868  |0.0455  |0.044   |NULL    |NULL    |125       |
|5983822|1999 |14564|1998   |US     |TX     |569900  |2      |NULL  |114   |5  |55    |200  |0       |0.995   |NULL   |0.7201  |NULL    |12.45   |0.0     |0.0     |NULL    |NULL    |103       |
|6008204|1999 |14606|1998   |U